# Evaluations

This notebook runs the FAQ agent evaluation and writes a README-ready evaluation section.

What it does:

1. Rebuilds the DataTalksClub FAQ search index.
2. Creates two agent prompt versions: `faq_agent_v1` and `faq_agent_v2`.
3. Loads evaluation questions from `eval/test_questions.json` if it exists.
4. Runs both agents on the same questions.
5. Uses an LLM judge to score the answers.
6. Saves results to CSV and writes `eval/README_evaluation_section.md`.

## Install dependencies once

Run this in your terminal if something is missing:

```bash
uv add requests python-frontmatter minsearch pydantic-ai pandas tqdm
```

Also make sure your `OPENAI_API_KEY` is available in the environment.

In [ ]:
from pathlib import Path
from datetime import datetime
import io
import json
import random
import secrets
import zipfile

import frontmatter
import pandas as pd
import requests
from minsearch import Index
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.messages import ModelMessagesTypeAdapter
from tqdm.auto import tqdm

# Make paths work whether this notebook is run from project root or from eval/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "eval":
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_DIR = PROJECT_ROOT / "eval"
LOG_DIR = PROJECT_ROOT / "logs"

EVAL_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Eval dir:", EVAL_DIR)
print("Log dir:", LOG_DIR)

## 1. Rebuild the FAQ search index

This repeats the setup from the main notebook so this evaluation notebook can run on its own.

In [ ]:
def read_repo_data(repo_owner: str, repo_name: str) -> list[dict]:
    """Download and parse markdown/mdx files from a GitHub repository."""
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for file_info in zf.infolist():
            filename = file_info.filename
            filename_lower = filename.lower()

            if not (filename_lower.endswith(".md") or filename_lower.endswith(".mdx")):
                continue

            try:
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode("utf-8", errors="ignore")
                    post = frontmatter.loads(content)
                    data = post.to_dict()
                    data["filename"] = filename
                    repository_data.append(data)
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    return repository_data


dtc_faq = read_repo_data("DataTalksClub", "faq")
de_dtc_faq = [d for d in dtc_faq if "data-engineering" in d["filename"]]

faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[],
)

faq_index.fit(de_dtc_faq)

print(f"Loaded {len(de_dtc_faq)} FAQ records")

In [ ]:
def text_search(query: str) -> list[dict]:
    """Search the course FAQ database."""
    return faq_index.search(query, num_results=5)

## 2. Create two prompt versions to compare

In [ ]:
system_prompt_v1 = """
You are a helpful assistant for the DataTalksClub course FAQ.

Use the search tool when answering course-related questions.
Answer clearly and briefly.
""".strip()


system_prompt_v2 = """
You are a careful assistant for the DataTalksClub course FAQ.

Always use the search tool before answering.
Only answer based on the search results.
If the search results do not contain the answer, say that you could not find enough information.

Always include citations using the filename of the source you used.
""".strip()


agent_v1 = Agent(
    name="faq_agent_v1",
    instructions=system_prompt_v1,
    tools=[text_search],
    model="gpt-4o-mini",
)

agent_v2 = Agent(
    name="faq_agent_v2",
    instructions=system_prompt_v2,
    tools=[text_search],
    model="gpt-4o-mini",
)

## 3. Logging helpers

Every test interaction gets saved as a JSON file in `logs/`.

In [ ]:
def _agent_tools(agent) -> list[str]:
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())
    return tools


def _agent_instructions(agent) -> str:
    return getattr(agent, "_instructions", "")


def _agent_provider(agent) -> str:
    try:
        return agent.model.system
    except Exception:
        return ""


def _agent_model_name(agent) -> str:
    try:
        return agent.model.model_name
    except Exception:
        return ""


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_entry(agent, question: str, result, source: str = "user") -> dict:
    messages = result.new_messages()
    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": _agent_instructions(agent),
        "provider": _agent_provider(agent),
        "model": _agent_model_name(agent),
        "tools": _agent_tools(agent),
        "question": question,
        "response": result.output,
        "messages": dict_messages,
        "source": source,
    }


def log_interaction_to_file(agent, question: str, result, source: str = "user") -> Path:
    entry = log_entry(agent=agent, question=question, result=result, source=source)

    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer, ensure_ascii=False)

    return filepath

## 4. Load evaluation questions

Run `eval/data-gen.ipynb` first if you want AI-generated questions. If `eval/test_questions.json` is missing, this notebook uses the manual fallback questions.

In [ ]:
manual_questions = [
    "I just discovered the course. Can I still join?",
    "How do I install Kafka in Python?",
    "Do I need Docker for the course?",
    "Where can I find the homework?",
    "What should I do if I missed a deadline?",
    "Can I use Windows for the course?",
    "How do I submit homework?",
    "Is there a certificate?",
    "What happens if the search results do not answer my question?",
    "Can I use Python 3.12 for the course?",
]

questions_path = EVAL_DIR / "test_questions.json"

if questions_path.exists():
    question_data = json.loads(questions_path.read_text(encoding="utf-8"))
    test_questions = question_data.get("test_questions", manual_questions)
    print(f"Loaded {len(test_questions)} questions from {questions_path}")
else:
    test_questions = manual_questions
    print(f"No {questions_path} found. Using {len(test_questions)} manual questions.")

test_questions

## 5. Run both agents on the full test set

In [ ]:
# Optional: set this to True if you want to delete previous prompt-comparison logs before running.
CLEAR_OLD_PROMPT_COMPARISON_LOGS = True

if CLEAR_OLD_PROMPT_COMPARISON_LOGS:
    for old_log in LOG_DIR.glob("faq_agent_v*.json"):
        try:
            old_record = json.loads(old_log.read_text(encoding="utf-8"))
            if old_record.get("source") == "prompt-comparison":
                old_log.unlink()
        except Exception:
            pass

for agent_to_test in [agent_v1, agent_v2]:
    for q in tqdm(test_questions, desc=f"Testing {agent_to_test.name}"):
        result = await agent_to_test.run(user_prompt=q)

        log_interaction_to_file(
            agent=agent_to_test,
            question=q,
            result=result,
            source="prompt-comparison",
        )

print(f"Finished testing {len(test_questions)} questions on 2 agents.")

## 6. Build the LLM judge

The judge checks whether each answer followed the prompt, used search, cited sources when required, and answered clearly.

In [ ]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met.

Checklist:

- instructions_follow: The agent followed the user's instructions in <INSTRUCTIONS>
- instructions_avoid: The agent avoided doing things it was told not to do
- answer_relevant: The response directly addresses the user's question
- answer_clear: The answer is clear and correct
- answer_citations: The response includes proper citations or sources when required
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: The search tool was invoked

Output true/false for each check and provide a short explanation for your judgment.
""".strip()


class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool


class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


# Use gpt-4o-mini if gpt-5-nano is not available in your account.
eval_agent = Agent(
    name="eval_agent",
    model="gpt-4o-mini",
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist,
)


user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

## 7. Evaluate the logged interactions

In [ ]:
def load_log_file(log_file: Path) -> dict:
    with open(log_file, "r", encoding="utf-8") as f_in:
        log_data = json.load(f_in)
        log_data["log_file"] = log_file
        return log_data


def simplify_log_messages(messages: list[dict]) -> list[dict]:
    simplified = []

    for message in messages:
        new_message = {
            "kind": message.get("kind"),
            "parts": [],
        }

        for original_part in message.get("parts", []):
            part = original_part.copy()

            # Remove noisy/expensive metadata
            part.pop("timestamp", None)
            part.pop("tool_call_id", None)
            part.pop("metadata", None)
            part.pop("id", None)

            # Replace actual search results to save tokens
            if part.get("part_kind") == "tool-return":
                part["content"] = "SEARCH_RESULTS_REDACTED"

            new_message["parts"].append(part)

        simplified.append(new_message)

    return simplified


async def evaluate_log_record(eval_agent, log_record: dict) -> EvaluationChecklist:
    instructions = log_record["system_prompt"]
    question = log_record["question"]
    answer = log_record["response"]

    log_simplified = simplify_log_messages(log_record["messages"])
    log = json.dumps(log_simplified, ensure_ascii=False)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log,
    )

    result = await eval_agent.run(user_prompt)
    return result.output

In [ ]:
eval_rows = []

log_files = sorted(LOG_DIR.glob("faq_agent_v*.json"))

for log_file in tqdm(log_files, desc="Evaluating logs"):
    log_record = load_log_file(log_file)

    if log_record.get("source") != "prompt-comparison":
        continue

    eval_result = await evaluate_log_record(eval_agent, log_record)

    row = {
        "file": log_file.name,
        "agent_name": log_record["agent_name"],
        "source": log_record["source"],
        "question": log_record["question"],
        "answer": log_record["response"],
        "summary": eval_result.summary,
    }

    for check in eval_result.checklist:
        row[check.check_name] = check.check_pass

    eval_rows.append(row)

df_evals = pd.DataFrame(eval_rows)

results_path = EVAL_DIR / "evaluation_results.csv"
df_evals.to_csv(results_path, index=False)

print(f"Saved detailed results to: {results_path}")
df_evals

## 8. Calculate scores

In [ ]:
score_columns = df_evals.select_dtypes(include="bool").columns

prompt_scores = df_evals.groupby("agent_name")[list(score_columns)].mean()
prompt_scores["overall_score"] = prompt_scores.mean(axis=1)

prompt_scores = prompt_scores.sort_values("overall_score", ascending=False)

scores_path = EVAL_DIR / "prompt_scores.csv"
prompt_scores.to_csv(scores_path)

print(f"Saved prompt scores to: {scores_path}")
prompt_scores

In [ ]:
best_agent = prompt_scores["overall_score"].idxmax()
best_score = prompt_scores["overall_score"].max()

print(f"The better prompt was {best_agent} with an overall score of {best_score:.2f}")

## 9. Write the README-ready evaluation section

Copy the generated `eval/README_evaluation_section.md` content into your main `README.md`.

In [ ]:
def make_readme_evaluation_section(prompt_scores: pd.DataFrame, best_agent: str, best_score: float) -> str:
    rows = []
    for agent_name, row in prompt_scores.iterrows():
        rows.append(f"| `{agent_name}` | {row['overall_score']:.2f} |")

    table = "
".join(rows)

    return f"""## Evaluation

I evaluated two versions of my FAQ agent:

- `faq_agent_v1`: simple helpful assistant prompt
- `faq_agent_v2`: stricter prompt requiring search usage and citations

I used manual questions and AI-generated questions. Each answer was logged and then evaluated with an LLM judge using a checklist.

The checklist evaluated:

- instruction following
- relevance
- clarity
- citation/source usage
- completeness
- search tool usage

### Results

| Agent | Overall Score |
|---|---:|
{table}

The better prompt was `{best_agent}` with an overall score of `{best_score:.2f}`.
"""


readme_section = make_readme_evaluation_section(prompt_scores, best_agent, best_score)
print(readme_section)

readme_section_path = EVAL_DIR / "README_evaluation_section.md"
readme_section_path.write_text(readme_section, encoding="utf-8")

print(f"
Saved README section to: {readme_section_path}")